<a href="https://colab.research.google.com/github/Pinuig/machine-learning-lab/blob/main/house_prices_advanced_regression_techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install xgboost lightgbm

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import skew

try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_FOLDER_NAME = "house-prices-advanced-regression-techniques"
    DATA_DIR = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}")

except ImportError:
    DATA_DIR = Path(".")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
OUTPUT_PATH = DATA_DIR / "submission.csv"

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Could not find {TRAIN_PATH}. Please verify your Drive folder name.")

print(f"Reading dataset from: {DATA_DIR.resolve()}")


from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

N_FOLDS = 10
RANDOM_STATE = 42

print("1. Loading Data & Removing Outliers...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train = train.drop(train[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)].index)
train = train.reset_index(drop=True)

y = np.log1p(train["SalePrice"].values)
train_features = train.drop(columns=["Id", "SalePrice"])
test_features = test.drop(columns=["Id"])
test_ids = test["Id"]

n_train = len(train_features)
all_data = pd.concat([train_features, test_features], axis=0, ignore_index=True)

print("2. Imputing Missing Values...")
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType"
]
for col in none_cols:
    all_data[col] = all_data[col].fillna("None")

zero_cols = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"
]
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))
all_data["MSZoning"] = all_data.groupby("MSSubClass")["MSZoning"].transform(lambda x: x.fillna(x.mode()[0]))
all_data["Functional"] = all_data["Functional"].fillna("Typ")
all_data["Utilities"] = all_data["Utilities"].fillna(all_data["Utilities"].mode()[0])

for col in all_data.columns:
    if all_data[col].isnull().sum() > 0:
        if all_data[col].dtype == "object":
            all_data[col] = all_data[col].fillna(all_data[col].mode()[0])
        else:
            all_data[col] = all_data[col].fillna(all_data[col].median())

print("3. Advanced Feature Engineering...")
qual_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
qual_cols = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
for col in qual_cols:
    all_data[col] = all_data[col].map(qual_map).astype(int)

all_data["BsmtExposure"] = all_data["BsmtExposure"].map({"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}).astype(int)
all_data["BsmtFinType1"] = all_data["BsmtFinType1"].map({"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}).astype(int)
all_data["GarageFinish"] = all_data["GarageFinish"].map({"None": 0, "Unf": 1, "RFn": 2, "Fin": 3}).astype(int)
all_data["MSSubClass"] = all_data["MSSubClass"].astype(str)

all_data["TotalSF"] = all_data["TotalBsmtSF"] + all_data["1stFlrSF"] + all_data["2ndFlrSF"]
all_data["TotalBathrooms"] = all_data["FullBath"] + (0.5 * all_data["HalfBath"]) + all_data["BsmtFullBath"] + (0.5 * all_data["BsmtHalfBath"])
all_data["HouseAge"] = all_data["YrSold"] - all_data["YearBuilt"]
all_data["RemodAge"] = all_data["YrSold"] - all_data["YearRemodAdd"]
all_data["TotalPorchSF"] = all_data["OpenPorchSF"] + all_data["EnclosedPorch"] + all_data["3SsnPorch"] + all_data["ScreenPorch"]

all_data["OverallQual_TotalSF"] = all_data["OverallQual"] * all_data["TotalSF"]
all_data["GrLivArea_OverallQual"] = all_data["GrLivArea"] * all_data["OverallQual"]
all_data["ExterQual_TotalSF"] = all_data["ExterQual"] * all_data["TotalSF"]
all_data["KitchenQual_TotalSF"] = all_data["KitchenQual"] * all_data["TotalSF"]

numeric_cols = all_data.select_dtypes(include=[np.number]).columns
skewed = all_data[numeric_cols].apply(lambda s: skew(s)).sort_values(ascending=False)
high_skew = skewed[abs(skewed) > 0.75].index

for col in high_skew:
    all_data[col] = np.log1p(all_data[col])

all_data = pd.get_dummies(all_data).astype(float)

X = all_data.iloc[:n_train, :].values
X_test = all_data.iloc[n_train:, :].values

print(f"Processed dataset shape: {X.shape}")

print("4. Out-of-Fold Model Training & Evaluation...")
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

models = {
    "ridge": make_pipeline(RobustScaler(), Ridge(alpha=15.0)),
    "lasso": make_pipeline(RobustScaler(), Lasso(alpha=0.0005, random_state=RANDOM_STATE)),
    "enet": make_pipeline(RobustScaler(), ElasticNet(alpha=0.0005, l1_ratio=0.9, random_state=RANDOM_STATE)),
    "svr": make_pipeline(RobustScaler(), SVR(C=20, epsilon=0.008, gamma=0.0003)),
}

if HAS_XGB:
    models["xgb"] = XGBRegressor(
        n_estimators=2200, learning_rate=0.015, max_depth=3,
        subsample=0.6, colsample_bytree=0.6, reg_alpha=0.5, reg_lambda=0.6,
        random_state=RANDOM_STATE, n_jobs=-1
    )

if HAS_LGB:
    models["lgb"] = LGBMRegressor(
        n_estimators=1800, learning_rate=0.015, max_depth=3,
        num_leaves=8, subsample=0.6, colsample_bytree=0.6,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )

weights = {
    "ridge": 0.10,
    "lasso": 0.15,
    "enet": 0.15,
    "svr": 0.15,
    "xgb": 0.25 if HAS_XGB else 0.0,
    "lgb": 0.20 if HAS_LGB else 0.0,
}

total_w = sum(w for k, w in weights.items() if k in models)
weights = {k: v / total_w for k, v in weights.items() if k in models}

oof_preds = np.zeros((len(X), len(models)))
test_preds = np.zeros((len(X_test), len(models)))

for m_idx, (name, model) in enumerate(models.items()):
    oof = np.zeros(len(X))
    preds = np.zeros(len(X_test))

    for train_idx, val_idx in kf.split(X, y):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_va, y_va = X[val_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        oof[val_idx] = model.predict(X_va)
        preds += model.predict(X_test) / N_FOLDS

    rmse = np.sqrt(np.mean((y - oof) ** 2))
    print(f"  --> {name.upper():<8} OOF RMSE: {rmse:.5f}")

    oof_preds[:, m_idx] = oof
    test_preds[:, m_idx] = preds

blended_oof = np.zeros(len(X))
blended_test = np.zeros(len(X_test))

for m_idx, name in enumerate(models.keys()):
    blended_oof += oof_preds[:, m_idx] * weights[name]
    blended_test += test_preds[:, m_idx] * weights[name]

final_rmse = np.sqrt(np.mean((y - blended_oof) ** 2))
print(f"\n----------------------------------------")
print(f"FINAL BLENDED OOF RMSE: {final_rmse:.5f}")
print(f"------------------------------------------")

submission = pd.DataFrame({"Id": test_ids, "SalePrice": np.expm1(blended_test)})
submission.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved submission to: {OUTPUT_PATH.resolve()}")

Mounted at /content/drive
Reading dataset from: /content/drive/MyDrive/house-prices-advanced-regression-techniques
1. Loading Data & Removing Outliers...
2. Imputing Missing Values...
3. Advanced Feature Engineering...
Processed dataset shape: (1458, 272)
4. Out-of-Fold Model Training & Evaluation...
  --> RIDGE    OOF RMSE: 0.11210


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.885e-02, tolerance: 2.087e-02
  model = cd_fast.enet_coordinate_descent(


  --> LASSO    OOF RMSE: 0.11135


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.921e-02, tolerance: 2.087e-02
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.382e+00, tolerance: 2.120e-02
  model = cd_fast.enet_coordinate_descent(


  --> ENET     OOF RMSE: 0.11137
  --> SVR      OOF RMSE: 0.11209
  --> XGB      OOF RMSE: 0.11303


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  --> LGB      OOF RMSE: 0.11744

----------------------------------------
FINAL BLENDED OOF RMSE: 0.10808
------------------------------------------

Saved submission to: /content/drive/MyDrive/house-prices-advanced-regression-techniques/submission.csv


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
from pathlib import Path

drive_folder = Path("/content/drive/MyDrive/house-prices-advanced-regression-techniques")
kaggle_dir = Path.home() / ".kaggle"

# Create hidden .kaggle directory
kaggle_dir.mkdir(exist_ok=True)

token_src = drive_folder / "kaggle.json"
token_dst = kaggle_dir / "kaggle.json"

if token_src.exists():
    !cp "{token_src}" "{token_dst}"
    !chmod 600 "{token_dst}"
    print("Kaggle API successfully configured!")
else:
    print(f"Error: kaggle.json missing from {token_src}")

Kaggle API successfully configured!


In [ ]:
sub_path = drive_folder / "submission.csv"

!kaggle competitions submit \
  -c house-prices-advanced-regression-techniques \
  -f "{sub_path}" \
  -m "Blended OOF model submission (Score ~0.10808)"

100% 33.7k/33.7k [00:00<00:00, 42.3kB/s]
Successfully submitted to House Prices - Advanced Regression Techniques

In [ ]:
!kaggle competitions submissions -c house-prices-advanced-regression-techniques

fileName        date                        description                                    status                     publicScore  privateScore  
--------------  --------------------------  ---------------------------------------------  -------------------------  -----------  ------------  
submission.csv  2026-08-07 11:24:19.130000  Blended OOF model submission (Score ~0.10808)  SubmissionStatus.COMPLETE  0.12088                    


In [ ]:
!pip install catboost -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

!pip install catboost lightgbm xgboost -q

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import skew
from scipy.special import boxcox1p
from scipy.optimize import minimize

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_FOLDER_NAME = "house-prices-advanced-regression-techniques"
    DATA_DIR = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}")
except ImportError:
    DATA_DIR = Path(".")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
OUTPUT_PATH = DATA_DIR / "submission_optimized_ensemble.csv"

N_FOLDS = 10
RANDOM_STATE = 42

print("1. Loading Data & Removing Outliers...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train = train.drop(train[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)].index).reset_index(drop=True)

y = np.log1p(train["SalePrice"].values)
train_features = train.drop(columns=["Id", "SalePrice"])
test_features = test.drop(columns=["Id"])
test_ids = test["Id"]

n_train = len(train_features)
all_data = pd.concat([train_features, test_features], axis=0, ignore_index=True)

print("2. Imputing Missing Values & Encoding Categoricals...")

none_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu", "GarageType",
             "GarageFinish", "GarageQual", "GarageCond", "BsmtQual", "BsmtCond",
             "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "MasVnrType"]
for col in none_cols:
    all_data[col] = all_data[col].fillna("None")

zero_cols = ["GarageYrBlt", "GarageArea", "GarageCars", "BsmtFinSF1",
             "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath",
             "BsmtHalfBath", "MasVnrArea"]
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))
all_data["MSZoning"] = all_data.groupby("MSSubClass")["MSZoning"].transform(lambda x: x.fillna(x.mode()[0]))
all_data["Functional"] = all_data["Functional"].fillna("Typ")
all_data["Utilities"] = all_data["Utilities"].fillna(all_data["Utilities"].mode()[0])
all_data["Electrical"] = all_data["Electrical"].fillna(all_data["Electrical"].mode()[0])
all_data["KitchenQual"] = all_data["KitchenQual"].fillna(all_data["KitchenQual"].mode()[0])
all_data["Exterior1st"] = all_data["Exterior1st"].fillna(all_data["Exterior1st"].mode()[0])
all_data["Exterior2nd"] = all_data["Exterior2nd"].fillna(all_data["Exterior2nd"].mode()[0])
all_data["SaleType"] = all_data["SaleType"].fillna(all_data["SaleType"].mode()[0])

all_data["MSSubClass"] = all_data["MSSubClass"].astype(str)
all_data["MoSold"] = all_data["MoSold"].astype(str)

qual_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
qual_cols = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
             "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
for col in qual_cols:
    all_data[col] = all_data[col].map(qual_map).astype(int)

all_data["BsmtExposure"] = all_data["BsmtExposure"].map({"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}).astype(int)
all_data["BsmtFinType1"] = all_data["BsmtFinType1"].map({"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}).astype(int)
all_data["BsmtFinType2"] = all_data["BsmtFinType2"].map({"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}).astype(int)
all_data["GarageFinish"] = all_data["GarageFinish"].map({"None": 0, "Unf": 1, "RFn": 2, "Fin": 3}).astype(int)

print("3. Feature Engineering & Transformations...")

garage_yr = all_data["GarageYrBlt"].where(all_data["GarageYrBlt"] > 0, all_data["YearBuilt"])

all_data["TotalSF"] = all_data["TotalBsmtSF"] + all_data["1stFlrSF"] + all_data["2ndFlrSF"]
all_data["TotalBathrooms"] = all_data["FullBath"] + (0.5 * all_data["HalfBath"]) + all_data["BsmtFullBath"] + (0.5 * all_data["BsmtHalfBath"])
all_data["HouseAge"] = all_data["YrSold"] - all_data["YearBuilt"]
all_data["RemodAge"] = all_data["YrSold"] - all_data["YearRemodAdd"]
all_data["GarageAge"] = all_data["YrSold"] - garage_yr
all_data["IsRemodeled"] = (all_data["YearBuilt"] != all_data["YearRemodAdd"]).astype(int)
all_data["TotalPorchSF"] = all_data["OpenPorchSF"] + all_data["EnclosedPorch"] + all_data["3SsnPorch"] + all_data["ScreenPorch"]

all_data["TotalOverallQual"] = all_data["OverallQual"] + all_data["OverallCond"]
all_data["TotalStructureQual"] = (all_data["ExterQual"] + all_data["BsmtQual"] +
                                  all_data["KitchenQual"] + all_data["HeatingQC"] +
                                  all_data["GarageQual"])

all_data["OverallQual_TotalSF"] = all_data["OverallQual"] * all_data["TotalSF"]
all_data["GrLivArea_OverallQual"] = all_data["GrLivArea"] * all_data["OverallQual"]
all_data["Qual_Cond_Product"] = all_data["OverallQual"] * all_data["OverallCond"]

all_data["HasPool"] = (all_data["PoolArea"] > 0).astype(int)
all_data["Has2ndFloor"] = (all_data["2ndFlrSF"] > 0).astype(int)
all_data["HasGarage"] = (all_data["GarageArea"] > 0).astype(int)
all_data["HasBsmt"] = (all_data["TotalBsmtSF"] > 0).astype(int)
all_data["HasFireplace"] = (all_data["Fireplaces"] > 0).astype(int)

all_data["Neigh_Qual_Mean"] = all_data.groupby("Neighborhood")["OverallQual"].transform("mean")
all_data["Neigh_TotalSF_Mean"] = all_data.groupby("Neighborhood")["TotalSF"].transform("mean")

continuous_cols = [
    "LotFrontage", "LotArea", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", "GarageArea",
    "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea",
    "MiscVal", "TotalSF", "TotalPorchSF"
]

skewed_feats = all_data[continuous_cols].apply(lambda s: skew(s)).sort_values(ascending=False)
high_skew = skewed_feats[abs(skewed_feats) > 0.75].index
for col in high_skew:
    all_data[col] = boxcox1p(all_data[col], 0.15)

all_data = pd.get_dummies(all_data).astype(float)

X = all_data.iloc[:n_train, :].values
X_test = all_data.iloc[n_train:, :].values
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print("\n4. Initializing Base Models...")

base_models = {
    "ridge": make_pipeline(RobustScaler(), Ridge(alpha=15.0, max_iter=10000)),
    "lasso": make_pipeline(RobustScaler(), Lasso(alpha=0.00045, max_iter=20000, random_state=RANDOM_STATE)),
    "enet": make_pipeline(RobustScaler(), ElasticNet(alpha=0.0007, l1_ratio=0.6, max_iter=20000, random_state=RANDOM_STATE)),
    "bayesian": make_pipeline(RobustScaler(), BayesianRidge(max_iter=1000)),
    "krr": make_pipeline(RobustScaler(), KernelRidge(alpha=0.6, kernel='polynomial', degree=2, coef0=2.5)),
    "svr": make_pipeline(RobustScaler(), SVR(C=20, epsilon=0.008, gamma=0.0003)),
    "ext": ExtraTreesRegressor(n_estimators=1000, max_depth=10, max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1),
    "gbr": GradientBoostingRegressor(n_estimators=3000, learning_rate=0.01, max_depth=3,
                                     max_features='sqrt', min_samples_leaf=15, min_samples_split=10,
                                     loss='huber', random_state=RANDOM_STATE),
    "xgb": XGBRegressor(n_estimators=2600, learning_rate=0.01, max_depth=3, subsample=0.6,
                        colsample_bytree=0.4, reg_alpha=0.4, reg_lambda=0.6,
                        random_state=RANDOM_STATE, n_jobs=-1),
    "lgb": LGBMRegressor(n_estimators=2200, learning_rate=0.01, max_depth=3, num_leaves=7,
                         subsample=0.6, colsample_bytree=0.4, reg_alpha=0.3, reg_lambda=0.5,
                         random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    "cat": CatBoostRegressor(iterations=2500, learning_rate=0.01, depth=4, l2_leaf_reg=3.0,
                             random_seed=RANDOM_STATE, verbose=0)
}

oof_train = np.zeros((len(X), len(base_models)))
oof_test_folds = np.zeros((len(X_test), len(base_models), N_FOLDS))

print("\n5. Training Base Models & Generating OOF Predictions...")

for m_idx, (name, model) in enumerate(base_models.items()):
    oof = np.zeros(len(X))
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_va, y_va = X[val_idx], y[val_idx]
        model.fit(X_tr, y_tr)
        oof[val_idx] = model.predict(X_va)
        oof_test_folds[:, m_idx, fold_idx] = model.predict(X_test)

    oof_train[:, m_idx] = oof
    rmse = np.sqrt(np.mean((y - oof) ** 2))
    print(f"  --> Base Model {name.upper():<10} OOF RMSE: {rmse:.5f}")

oof_test = oof_test_folds.mean(axis=2)

print("\n6. Finding Optimal Non-Negative Ensembling Weights (SLSQP)...")

def rmse_obj(weights):
    blend_pred = oof_train @ weights
    return np.sqrt(np.mean((y - blend_pred) ** 2))

num_models = len(base_models)
init_weights = np.ones(num_models) / num_models
bounds = [(0, 1) for _ in range(num_models)]
constraints = ({'type': 'eq', 'fun': lambda w: 1 - np.sum(w)})

res = minimize(rmse_obj, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)

opt_weights = res.x
slsqp_oof_rmse = res.fun

print("\n----------------------------------------------------------------")
print(f"SLSQP OPTIMIZED ENSEMBLE OOF RMSE: {slsqp_oof_rmse:.5f}")
print("----------------------------------------------------------------")
print("Learned Model Weights:")
for name, weight in zip(base_models.keys(), opt_weights):
    print(f"  {name.upper():<10}: {weight * 100:6.2f}%")

meta_model = Lasso(alpha=0.0001, max_iter=10000, random_state=RANDOM_STATE)
meta_oof = np.zeros(len(X))
meta_test_preds = np.zeros(len(X_test))

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(oof_train, y)):
    meta_model.fit(oof_train[train_idx], y[train_idx])
    meta_oof[val_idx] = meta_model.predict(oof_train[val_idx])
    meta_test_preds += meta_model.predict(oof_test) / N_FOLDS

meta_oof_rmse = np.sqrt(np.mean((y - meta_oof) ** 2))
print(f"META-LASSO STACKING OOF RMSE    : {meta_oof_rmse:.5f}")

slsqp_test_preds = oof_test @ opt_weights
final_test_preds = 0.60 * slsqp_test_preds + 0.40 * meta_test_preds

submission = pd.DataFrame({"Id": test_ids, "SalePrice": np.expm1(final_test_preds)})
submission.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved optimized ensemble submission to: {OUTPUT_PATH.resolve()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. Loading Data & Removing Outliers...
2. Imputing Missing Values & Encoding Categoricals...
3. Feature Engineering & Transformations...

4. Initializing Base Models...

5. Training Base Models & Generating OOF Predictions...
  --> Base Model RIDGE      OOF RMSE: 0.11055
  --> Base Model LASSO      OOF RMSE: 0.10948
  --> Base Model ENET       OOF RMSE: 0.10955
  --> Base Model BAYESIAN   OOF RMSE: 0.11057
  --> Base Model KRR        OOF RMSE: 0.11186
  --> Base Model SVR        OOF RMSE: 0.10961
  --> Base Model EXT        OOF RMSE: 0.14051
  --> Base Model GBR        OOF RMSE: 0.11062
  --> Base Model XGB        OOF RMSE: 0.11028
  --> Base Model LGB        OOF RMSE: 0.11550
  --> Base Model CAT        OOF RMSE: 0.11220

6. Finding Optimal Non-Negative Ensembling Weights (SLSQP)...

----------------------------------------------------------------
SLSQP OPTI

In [ ]:
!kaggle competitions submit \
  -c house-prices-advanced-regression-techniques \
  -f "/content/drive/MyDrive/house-prices-advanced-regression-techniques/submission.csv" \
  -m "Advanced Stacking Regressor Meta-Learner with Domain Feature Interactions"

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
